# GEAP Agent Gateway — Govern Traffic & Attach Policies (SDK-First Deep-Dive)

A companion to `platform_sdk_demo.ipynb`. The **Agent Gateway** is the policy-enforcement point at the network boundary: it governs **ingress** (who may call an agent) and **egress** (what an agent may call — Gemini, MCP tools, external APIs), and it's where you attach IAM, Semantic Governance, IAP, and Model Armor policies. This notebook shows the gateway config live and the create/attach/policy steps as exact, guarded commands.

**Legend.** ✅ runs live here (in-process dict/JSON construction). 🔒 shown-but-guarded: the gcloud / REST call is printed and only executed when `GEAP_RUN_GATEWAY=1` (create/policies) or `GEAP_RUN_DEPLOY=1` (attach-at-deploy) is set. 🔧 marks custom/infra (not an ADK object).

> Gateway create + policy attachment need **Network Services enrollment** (Private Preview) and a deployed agent; attachment is set at `agent_engines.create` time and can't be PATCHed on. Headless twins: `bash scripts/setup_agent_gateway.sh` and `bash scripts/setup_governance_policies.sh --sgp`.

## Setup

In [1]:
import os, json, shlex, subprocess
for _ in range(6):
    if os.path.exists("src/config.py"):
        break
    os.chdir("..")

# Enable the gateway/identity toggles BEFORE importing config so the builders show BOTH modes.
_pid = os.environ.get("GCP_PROJECT_ID", "wortz-project-352116")
_reg = os.environ.get("GCP_REGION", "us-central1")
os.environ["ENABLE_AGENT_GATEWAY"] = "1"
os.environ["ENABLE_AGENT_IDENTITY"] = "1"
os.environ.setdefault("AGENT_GATEWAY_EGRESS_PATH",
    f"projects/{_pid}/locations/{_reg}/agentGateways/geap-workshop-gateway-egress")

from src.config import GCP_PROJECT_ID, GCP_REGION

RUN_GATEWAY = os.environ.get("GEAP_RUN_GATEWAY") == "1"   # create gateways + attach policies
RUN_DEPLOY  = os.environ.get("GEAP_RUN_DEPLOY") == "1"    # re-deploy the agent to attach the gateway

def run_or_show(cmd, live: bool):
    """🔧 Print a command; execute it (never raising) only when `live` is True."""
    printable = cmd if isinstance(cmd, str) else " ".join(shlex.quote(c) for c in cmd)
    print(f"$ {printable}")
    if not live:
        print("  🔒 skipped — set the guard env var to run this for real")
        return None
    try:
        out = subprocess.run(cmd, shell=isinstance(cmd, str), capture_output=True, text=True, timeout=180)
        print((out.stdout or out.stderr)[-2000:])
        return out
    except Exception as e:
        print(f"  (error: {type(e).__name__}: {e})")
        return None

print("project:", GCP_PROJECT_ID, "| region:", GCP_REGION)
print("guards -> RUN_GATEWAY:", RUN_GATEWAY, "| RUN_DEPLOY:", RUN_DEPLOY)

project: wortz-project-352116 | region: us-central1
guards -> RUN_GATEWAY: False | RUN_DEPLOY: False


## Phase 1 — The dual-mode gateway config
📖 [`scripts/setup_agent_gateway.sh`](https://github.com/jswortz/geap-tour/blob/main/scripts/setup_agent_gateway.sh) · [workshop guide §2.1](https://github.com/jswortz/geap-tour/blob/main/docs/workshop_guide.md)

A gateway runs in one of two governed-access modes. **Ingress** (`CLIENT_TO_AGENT`) controls inbound calls to your agent; **egress** (`AGENT_TO_ANYWHERE`) routes ALL outbound agent traffic through governance. You attach both to an agent at deploy time. Here's the exact config `deploy_agents.py` builds (pure Python — no cloud call).

In [2]:
from src.deploy.deploy_agents import _build_gateway_config, _build_config
from types import SimpleNamespace

gw = _build_gateway_config()   # reads ENABLE_AGENT_GATEWAY + the two gateway paths
print("agent_gateway_config:")
print(json.dumps(gw, indent=2))

cfg = _build_config(SimpleNamespace(name="coordinator_agent"))   # full deploy config
print("\nidentity_type:", cfg.get("identity_type"))
print("gateway wired into deploy config:", "agent_gateway_config" in cfg)

agent_gateway_config:
{
  "agent_to_anywhere_config": {
    "agent_gateway": "projects/wortz-project-352116/locations/us-central1/agentGateways/geap-workshop-gateway-egress"
  },
  "client_to_agent_config": {
    "agent_gateway": "projects/wortz-project-352116/locations/us-central1/agentGateways/geap-workshop-gateway"
  }
}
  Identity: AGENT_IDENTITY (SPIFFE-based)
  Gateway: egress=projects/wortz-project-352116/locations/us-central1/agentGateways/geap-workshop-gateway-egress, ingress=projects/wortz-project-352116/locations/us-central1/agentGateways/geap-workshop-gateway

identity_type: IdentityType.AGENT_IDENTITY
gateway wired into deploy config: True


## Phase 2 — Create the gateways — 🔒 guarded
📖 [workshop guide §2.1](https://github.com/jswortz/geap-tour/blob/main/docs/workshop_guide.md)

Gateways are created via the `networkservices v1beta1` API (the workshop script wraps the REST calls). Each gateway declares `protocols:["MCP"]` and a `governedAccessPath`; an egress gateway also lists the `registries` it fronts. A single gateway can't serve both Agent Runtime (regional) and Gemini Enterprise (global) — the script creates one set for each.

In [3]:
# The create bodies (illustrative — what setup_agent_gateway.sh POSTs):
ingress_body = {"protocols": ["MCP"], "googleManaged": {"governedAccessPath": "CLIENT_TO_AGENT"}}
egress_body = {
    "protocols": ["MCP"],
    "googleManaged": {"governedAccessPath": "AGENT_TO_ANYWHERE"},
    "registries": [f"//agentregistry.googleapis.com/projects/{GCP_PROJECT_ID}/locations/{GCP_REGION}"],
}
print("ingress:", json.dumps(ingress_body))
print("egress :", json.dumps(egress_body))

# Create all four gateways (regional + global, ingress + egress) via the workshop script:
run_or_show("bash scripts/setup_agent_gateway.sh", RUN_GATEWAY)

ingress: {"protocols": ["MCP"], "googleManaged": {"governedAccessPath": "CLIENT_TO_AGENT"}}
egress : {"protocols": ["MCP"], "googleManaged": {"governedAccessPath": "AGENT_TO_ANYWHERE"}, "registries": ["//agentregistry.googleapis.com/projects/wortz-project-352116/locations/us-central1"]}
$ bash scripts/setup_agent_gateway.sh
  🔒 skipped — set the guard env var to run this for real


## Phase 3 — Attach the gateway at deploy — 🔒 guarded
📖 [`deploy_agents.py`](https://github.com/jswortz/geap-tour/blob/main/src/deploy/deploy_agents.py)

The gateway is bound to an agent through `agent_gateway_config` + `identity_type=AGENT_IDENTITY` in `agent_engines.create/update` — the config from Phase 1. It **must** be set at create/update; it can't be PATCHed onto a running agent. (Egress has a known Private-Preview limitation — see the caveats cell.)

In [4]:
# Re-deploy the coordinator with the gateway + agent identity attached:
run_or_show(
    "ENABLE_AGENT_GATEWAY=1 ENABLE_AGENT_IDENTITY=1 "
    "uv run python -m src.deploy.deploy_agents coordinator --update",
    RUN_DEPLOY,
)

$ ENABLE_AGENT_GATEWAY=1 ENABLE_AGENT_IDENTITY=1 uv run python -m src.deploy.deploy_agents coordinator --update
  🔒 skipped — set the guard env var to run this for real


## Phase 4 — Attach policies (three governance layers) — 🔒 guarded
📖 [`scripts/setup_governance_policies.sh`](https://github.com/jswortz/geap-tour/blob/main/scripts/setup_governance_policies.sh)

Defense in depth. **Layer 1 — IAM Allow (CEL)**: static, per-tool allow rules via IAP. **Layer 2 — Semantic Governance Policies (SGP)**: runtime, natural-language business rules (verdicts ALLOW / DENY / ALLOW_IF_CONFIRMED). **Layer 3 — Authorization delegation**: IAP (`REQUEST_AUTHZ`) and Model Armor (`CONTENT_AUTHZ`) attached as `authzExtensions` + `authzPolicies`. Note: **max 4 authz policies per gateway**.

In [5]:
# Layer 1 — IAM Allow via CEL (per-tool). Example: coordinator may only call READ-ONLY search tools.
cel = "api.getAttribute('iap.googleapis.com/mcp.tool.isReadOnly', false) == true"
print("Layer 1 (CEL condition):", cel)

# Layer 2 — a Semantic Governance Policy body (natural-language constraint, tool-scoped):
sgp_body = {
    "displayName": "geap-expense-limit",
    "agent": f"projects/{GCP_PROJECT_ID}/locations/{GCP_REGION}/agents/coordinator_agent",
    "naturalLanguageConstraint": "Never submit an expense over $5,000 without manager approval.",
}
print("Layer 2 (SGP body):", json.dumps(sgp_body))

# Layer 3 — an authz extension (Model Armor, CONTENT_AUTHZ) targeting the gateway:
authz_ext = {"service": f"modelarmor.{GCP_REGION}.rep.googleapis.com", "failOpen": False,
             "metadata": {"model_armor_settings": "[{request_template_id, response_template_id}]"}}
print("Layer 3 (authz extension):", json.dumps(authz_ext))

# Apply all three layers (IAM + IAP + Model Armor; add --sgp for Layer 2) via the workshop script:
run_or_show("bash scripts/setup_governance_policies.sh --sgp", RUN_GATEWAY)

Layer 1 (CEL condition): api.getAttribute('iap.googleapis.com/mcp.tool.isReadOnly', false) == true
Layer 2 (SGP body): {"displayName": "geap-expense-limit", "agent": "projects/wortz-project-352116/locations/us-central1/agents/coordinator_agent", "naturalLanguageConstraint": "Never submit an expense over $5,000 without manager approval."}
Layer 3 (authz extension): {"service": "modelarmor.us-central1.rep.googleapis.com", "failOpen": false, "metadata": {"model_armor_settings": "[{request_template_id, response_template_id}]"}}
$ bash scripts/setup_governance_policies.sh --sgp
  🔒 skipped — set the guard env var to run this for real


## Phase 5 — Inspect what's attached — 🔒 guarded (read-only)

Verify the gateway, its attachment to the agent, and the authz policies/extensions bound to it.

In [6]:
# The agent must show identityType=AGENT_IDENTITY and a non-empty agentGatewayConfig once attached.
run_or_show(["gcloud", "beta", "service-extensions", "authz-extensions", "list",
             "--location", GCP_REGION, "--project", GCP_PROJECT_ID], RUN_GATEWAY)
run_or_show(["gcloud", "beta", "network-security", "authz-policies", "list",
             "--location", GCP_REGION, "--project", GCP_PROJECT_ID], RUN_GATEWAY)

$ gcloud beta service-extensions authz-extensions list --location us-central1 --project wortz-project-352116
  🔒 skipped — set the guard env var to run this for real
$ gcloud beta network-security authz-policies list --location us-central1 --project wortz-project-352116
  🔒 skipped — set the guard env var to run this for real


## Caveats & Recap

- **Private Preview**: gateway attachment needs Network Services enrollment (separate from AI Platform enrollment). Without it, attach fails `code:13 INTERNAL`.
- **Egress limitation**: routing ALL outbound over the egress gateway currently breaks non-MCP gRPC calls (e.g. session creation) — see [`docs/gateway_test_report.md`](https://github.com/jswortz/geap-tour/blob/main/docs/gateway_test_report.md). Only MCP tool calls via `AgentRegistry.get_mcp_toolset()` traverse it cleanly.
- **Limits**: max **4** authz policies per gateway; ingress + global vs regional are mutually exclusive.

**Recap — what's what:** ✅ the dual-mode `agent_gateway_config` dict (`_build_gateway_config`); 🔒 create gateways (`setup_agent_gateway.sh`), attach at deploy (`deploy_agents --update`), and the 3 policy layers (`setup_governance_policies.sh --sgp`). Next: govern discovery across projects in **`registry_sdk_demo.ipynb`**.